# Нелінійні функції, softmax та attention

У попередній лабораторній ми побудували лінійний шар `linear(x, W, b)`:
$z = Wx + b$. Якщо скласти кілька таких шарів без функцій активації,
результат знову можна записати як один лінійний шар. Щоб мережа могла
моделювати складніші залежності, між шарами потрібна нелінійність.

Цього разу працюємо зі звичайними списками Python. Доповніть функції на
місці `...`, а потім запустіть `python3 attention.py`. Графіки до лабораторної
лежать у `figures/`.

## ReLU: пропускаємо додатні значення

$\operatorname{ReLU}(x)=\max(0,x)$. Для від'ємних чисел результат нульовий,
для додатних — саме число. Подивіться на `figures/activations.jpg`:
ReLU має злам у нулі. Що станеться з від'ємними компонентами вектора після
застосування ReLU?

![Графіки ReLU та GELU](figures/activations.jpg)

Нелінійність застосовується **після** лінійного шару, поелементно:
$h_i=\operatorname{ReLU}(z_i)$, де $z=Wx+b$.

In [ ]:
def relu(x):
    """Повертає ReLU одного числа."""
    ...


def test_relu():
    assert relu(-3) == 0
    assert relu(0) == 0
    assert relu(2.5) == 2.5


if __name__ == "__main__":
    test_relu()
    print("✓ relu")

## GELU: плавніша активація

GELU часто використовують у трансформерах. Її можна записати як
$\operatorname{GELU}(x)=x\Phi(x)$, де $\Phi$ — функція розподілу стандартної
нормальної величини. У Python її зручно обчислити через `math.erf`:

$$\operatorname{GELU}(x)=\frac{x}{2}
\left(1+\operatorname{erf}\left(\frac{x}{\sqrt{2}}\right)\right).$$

Порівняйте криві GELU та ReLU на графіку. GELU біля нуля плавна; для деяких
від'ємних $x$ вона теж повертає від'ємне, але мале число.

In [ ]:
def gelu(x):
    """Точна GELU через math.erf."""
    # Імпортуйте потрібні функції з math.
    ...


def test_gelu():
    from math import isclose

    assert gelu(0) == 0
    assert isclose(gelu(1), 0.8413447460685429, abs_tol=1e-12)
    assert isclose(gelu(-1), -0.15865525393145707, abs_tol=1e-12)
    assert isclose(gelu(3), 2.99595030590511, abs_tol=1e-12)


if __name__ == "__main__":
    test_gelu()
    print("✓ gelu")

## Поєднуємо активацію з лінійним шаром

Перенесіть або імпортуйте свою `linear` з першої лабораторної, якщо хочете
повторити обчислення. Тут уже готовий результат лінійного шару $z$.
Перетворіть кожен його елемент: спочатку через ReLU, потім через GELU.

```python
z = [-2.0, 0.0, 1.0]  # результат Wx + b
relu_vector(z)          # [0, 0, 1.0]
```

Питання: чому два лінійні шари поспіль можна згорнути в один, а два шари
з ReLU між ними — загалом ні?

In [ ]:
def relu_vector(values):
    """Поелементна ReLU для вектора."""
    ...


def gelu_vector(values):
    """Поелементна GELU для вектора."""
    ...


def test_activation_vectors():
    from math import isclose

    z = [-2.0, 0.0, 1.0]
    assert relu_vector(z) == [0, 0, 1.0]
    result = gelu_vector(z)
    assert len(result) == 3
    assert isclose(result[0], -0.04550026389635842, abs_tol=1e-12)
    assert result[1] == 0
    assert isclose(result[2], 0.8413447460685429, abs_tol=1e-12)
    assert z == [-2.0, 0.0, 1.0]


if __name__ == "__main__":
    test_activation_vectors()
    print("✓ activation vectors")

## Softmax: оцінки перетворюються на ваги

Для оцінок $s_1,\ldots,s_n$ softmax повертає
$p_i=e^{s_i}/\sum_j e^{s_j}$. Кожна вага додатна, а сума ваг дорівнює 1.
Softmax діє на **весь вектор**, на відміну від поелементних ReLU і GELU.
У багатокласовій класифікації модель спочатку видає довільні оцінки
(логіти) для кожного класу. Softmax перетворює їх на розподіл: числа від 0
до 1, сума яких **завжди дорівнює 1** для скінченних логітів. Тому їх часто
інтерпретують як імовірності класів. Проте сама нормалізація не гарантує,
що модель добре відкалібрована: вага 0.9 не обов'язково означає 90% успіху
на реальних даних. В attention ці самі числа — частки внеску значень, а не
ймовірності того, що певний ключ є «правильною відповіддю».

Наприклад, для оцінок класів `[0, ln(3)]` маємо ваги `[0.25, 0.75]`.
Другий клас отримує втричі більшу вагу, хоча різниця логітів — лише `ln(3)`.

На `figures/softmax.jpg` показано, як зміна однієї оцінки змінює всі ваги.
Спробуйте вручну обчислити softmax для `[0, 0]` та `[0, ln(3)]`.

![Ваги softmax для двох елементів](figures/softmax.jpg)

Але прямий виклик `exp(1000)` переповнює числа з плаваючою комою. Віднімання
одного й того самого числа від усіх оцінок не змінює softmax:

$$\frac{e^{s_i-c}}{\sum_j e^{s_j-c}}=
\frac{e^{s_i}}{\sum_j e^{s_j}}.$$

Оберіть $c=\max(s)$. Тоді найбільший показник експоненти дорівнює нулю,
а решта — від'ємні. Порожній вектор вважайте помилкою `ValueError`.
Припускаємо, що оцінки — скінченні числа.

In [ ]:
def softmax(scores):
    """Чисельно стійкий softmax для непорожнього вектора."""
    ...


def test_softmax():
    from math import isclose

    assert softmax([0, 0]) == [0.5, 0.5]
    assert softmax([0]) == [1.0]
    result = softmax([1, 2, 3])
    shifted = softmax([1001, 1002, 1003])
    assert all(isclose(a, b, abs_tol=1e-12) for a, b in zip(result, shifted))
    assert isclose(sum(result), 1.0, abs_tol=1e-12)
    assert softmax([1000, 1000]) == [0.5, 0.5]
    assert softmax([-1000, -1000]) == [0.5, 0.5]

    try:
        softmax([])
    except ValueError:
        pass
    else:
        raise AssertionError("Порожній вектор має давати ValueError")


if __name__ == "__main__":
    test_softmax()
    print("✓ softmax")

## Від лінійного шару до порівняння векторів

У першій лабораторній ми обчислювали `linear(x, W, b) = Wx + b`.
Для attention модель застосовує **різні навчені лінійні проєкції** до
вхідних векторів: з поточного вектора $x$ робить запит $q=W_Qx+b_Q$,
а з кожного доступного вектора $x_i$ — ключ $k_i=W_Kx_i+b_K$ і значення
$v_i=W_Vx_i+b_V$. Матриці $W_Q$, $W_K$ і $W_V$ — параметри моделі.
Запит і ключ мають однакову довжину $d_k$, щоб їх можна було порівняти.
Довжина значення може бути іншою.

Перший крок — `dot(q, k_i)`. Ось обчислення з функціями першої лабораторної
(одиничні проєкції й нульові зсуви для простоти):

```python
W_q = [[1, 0], [0, 1]]
W_k = [[1, 0], [0, 1]]
q = linear([1, 0], W_q, [0, 0])    # [1, 0]
k_1 = linear([1, 0], W_k, [0, 0])  # [1, 0]
k_2 = linear([0, 1], W_k, [0, 0])  # [0, 1]
dot(q, k_1), dot(q, k_2)           # (1, 0)
```

Перший ключ краще узгоджується із запитом.
Додатний добуток означає співнапрямленість, від'ємний — протилежність,
нульовий — ортогональність. Але це не відстань: довший вектор може дати
більший добуток лише через свій масштаб. Навчені проєкції дають моделі
змогу визначити, які ознаки порівнювати, а навчання підлаштовує і напрямки,
і масштаби.

Чому потрібні дві проєкції? Одна й та сама позиція може *шукати* одні
ознаки через свій запит і *пропонувати* інші через свій ключ. Наприклад,
у реченні поточне слово може шукати попередній іменник, а ключі попередніх
слів показують, яке з них відповідає цьому запиту.

## Attention для одного запиту

Тепер маємо один запит, кілька пар «ключ — значення» і три кроки.

1. **Порівняти:** `dot(q, k_i)` дає оцінку для кожного ключа. Ділимо її
   на $\sqrt{d_k}$, щоб за великої кількості координат оцінки не зростали
   надто сильно.
2. **Розподілити увагу:** softmax перетворює всі оцінки разом на невід'ємні
   ваги із сумою 1. Ключі з більшими оцінками отримують більший внесок,
   але кілька ключів можуть впливати одночасно. Це плавний, придатний до
   навчання спосіб розподілити «скільки дивитися» на кожну позицію.
3. **Зібрати інформацію:** кожне значення $v_i$ множимо на його вагу
   $\alpha_i$ і додаємо. Отримуємо новий вектор для поточної позиції.
   Саме значення переносить інформацію; ключ лише допомагає вирішити,
   яку частку цього значення використати.

Отже, це два різні множення. $q\cdot k_i$ відповідає на питання
«наскільки цей ключ підходить запиту?». Фінальне $\alpha V$ відповідає
«яку суміш значень взяти?». Для кожної координати результату це теж
скалярний добуток вектора ваг із відповідним стовпцем $V$, але вже не
порівняння подібності. Так можна розуміти attention як **перезважування
інформації**, яку отримує кожна позиція мережі.

$$s_i=\frac{q\cdot k_i}{\sqrt{d_k}},\qquad
\alpha=\operatorname{softmax}(s),\qquad
o=\sum_i\alpha_i v_i.$$

Розгляньмо `query = [1, 0]` і три ключі `[1, 0]`, `[0, 1]`, `[-1, 0]`.
Їхні скалярні добутки із запитом: `[1, 0, -1]`. Після ділення на
$\sqrt{2}$ та softmax ваги приблизно `[0.576, 0.284, 0.140]`.
Перший ключ узгоджується із запитом найбільше, але інші не зникають:
softmax дозволяє змішувати інформацію з кількох позицій. Якщо значення —
`[10, 0]`, `[0, 20]`, `[-10, 0]`, вихід приблизно `[4.36, 5.68]`.
Простежте кожен крок обчислення й перевірте зважену суму вручну.

Коли компоненти запиту й ключів мають приблизно одиничний масштаб,
дисперсія їхнього скалярного добутку зростає разом із $d_k$.
Великі за модулем оцінки роблять softmax надто різким; масштабування
стримує цей ефект. Для кількох запитів одразу отримуємо
$\operatorname{softmax}(QK^T/\sqrt{d_k})V$ (softmax у кожному рядку).
Цю форму scaled dot-product attention описано в статті
[Attention Is All You Need](https://arxiv.org/abs/1706.03762), розділ 3.2.1.

Реалізуйте `dot` тут або імпортуйте свою функцію з першої лабораторної.
Вимагайте однакову додатну довжину запиту та всіх ключів; ключів має бути
стільки ж, скільки значень. Усі значення повинні мати однакову додатну
довжину. За некоректних форм піднімайте `ValueError`.

In [ ]:
def dot(a, b):
    """Скалярний добуток; вектори різної довжини заборонені."""
    ...


def attention(query, keys, values):
    """Повертає (результат, ваги) для одного запиту."""
    # 1. Перевірте форми.
    # 2. Обчисліть оцінки через dot та поділіть на sqrt(len(query)).
    # 3. Отримайте ваги через softmax.
    # 4. Знайдіть зважену суму значень по кожній координаті.
    ...


def test_attention():
    from math import isclose

    keys = [[1, 0], [0, 1]]
    values = [[10, 0], [0, 20]]
    output, weights = attention([0, 0], keys, values)
    assert weights == [0.5, 0.5]
    assert output == [5.0, 10.0]

    output, weights = attention([1, 0], keys, values)
    assert weights[0] > weights[1]
    assert isclose(sum(weights), 1.0, abs_tol=1e-12)
    assert isclose(output[0], 10 * weights[0], abs_tol=1e-12)
    assert isclose(output[1], 20 * weights[1], abs_tol=1e-12)

    for bad_query, bad_keys, bad_values in [
        ([], keys, values),
        ([1, 0], [], []),
        ([1, 0], keys, [[1, 2]]),
        ([1, 0], [[1], [0, 1]], values),
        ([1, 0], keys, [[1], [2, 3]]),
    ]:
        try:
            attention(bad_query, bad_keys, bad_values)
        except ValueError:
            pass
        else:
            raise AssertionError("Некоректні форми мають давати ValueError")


if __name__ == "__main__":
    test_attention()
    print("✓ attention")

## Питання для обговорення

1. Чому softmax `[1000, 1001]` можна обчислити після віднімання 1001?
2. Що станеться з вагами attention, якщо помножити всі оцінки на велике число?
3. Чому `values` можуть мати іншу розмірність, ніж `keys`?
4. Які матричні операції з першої лабораторної допоможуть обчислити attention
   одразу для кількох запитів? Запишіть форми матриць $Q$, $K$, $V$,
   $QK^T$ та результату.